# Personal Intelligence Engine - Phase 2 OCR (Google Colab)

**Before running:** Make sure your PDFs are uploaded to Google Drive

In [ ]:
# Cell 1: Check GPU and Mount Drive
!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2: Find your PDFs - Let's search for them!
import os

print("Searching for notebooks_pdf folder in your Drive...\n")

# Common locations people upload to
search_paths = [
    "/content/drive/MyDrive/notebooks_pdf",
    "/content/drive/MyDrive/PersonalIntelligenceEngine/notebooks_pdf",
    "/content/drive/MyDrive/PortfolioProjects/PersonalIntelligenceEngine/notebooks_pdf",
]

found_path = None
for path in search_paths:
    if os.path.exists(path):
        print(f"✓ Found at: {path}")
        found_path = path
        break

if not found_path:
    print("❌ Not found in common locations.")
    print("\nLet's list your MyDrive to find it:\n")
    
    for item in os.listdir("/content/drive/MyDrive"):
        full_path = os.path.join("/content/drive/MyDrive", item)
        if os.path.isdir(full_path):
            print(f"📁 {item}")
            # Check 1 level deep
            try:
                for subitem in os.listdir(full_path)[:10]:  # Show first 10
                    print(f"   └── {subitem}")
            except:
                pass
    
    print("\n👉 PASTE THE CORRECT PATH BELOW:")
    print('PDF_DIR = "/content/drive/MyDrive/YOUR_FOLDER/notebooks_pdf"')
else:
    # List notebooks found
    notebooks = [d for d in os.listdir(found_path) if os.path.isdir(os.path.join(found_path, d))]
    print(f"\nFound {len(notebooks)} notebook(s):")
    for nb in notebooks:
        print(f"  📚 {nb}")
    
    # Set the path automatically
    PDF_DIR = found_path
    MD_DIR = found_path.replace("notebooks_pdf", "notebooks_md")
    os.makedirs(MD_DIR, exist_ok=True)
    
    print(f"\n✅ PDF_DIR set to: {PDF_DIR}")
    print(f"✅ MD_DIR set to: {MD_DIR}")

In [ ]:
# Cell 2.5: Check Structure & Rebuild if Needed
import os
import json

print("=" * 60)
print("Checking notebook structure...")
print("=" * 60)

# Check what's in the PDF directory
notebook_folders = [d for d in os.listdir(PDF_DIR) if os.path.isdir(os.path.join(PDF_DIR, d))]

print(f"\nFound {len(notebook_folders)} notebook folder(s):\n")

missing_structure = []
has_structure = []

for nb_folder in notebook_folders:
    nb_path = os.path.join(PDF_DIR, nb_folder)
    structure_file = os.path.join(nb_path, "_structure.json")
    
    if os.path.exists(structure_file):
        print(f"  ✓ {nb_folder}")
        print(f"    Has _structure.json")
        has_structure.append(nb_folder)
    else:
        print(f"  ⚠ {nb_folder}")
        print(f"    MISSING _structure.json")
        missing_structure.append((nb_folder, nb_path))
        
        # Show what PDFs are in this folder
        pdf_count = 0
        for root, dirs, files in os.walk(nb_path):
            pdf_count += len([f for f in files if f.endswith('.pdf')])
        print(f"    Contains {pdf_count} PDF file(s)")

if missing_structure:
    print(f"\n{'='*60}")
    print(f"⚠ WARNING: {len(missing_structure)} notebook(s) missing _structure.json")
    print("=" * 60)
    print("\nThe _structure.json files are created by Phase 1 extraction.")
    print("Without them, we need to rebuild the structure from your PDFs.\n")
    
    print("Rebuilding structure files...\n")
    
    for nb_name, nb_path in missing_structure:
        print(f"Building structure for: {nb_name}")
        
        # Find all PDFs in this notebook
        pages = []
        order = 0
        
        for root, dirs, files in os.walk(nb_path):
            # Get section name from folder structure
            rel_path = os.path.relpath(root, nb_path)
            section = rel_path if rel_path != '.' else 'Uncategorized'
            
            for file in sorted(files):
                if file.endswith('.pdf'):
                    pdf_path = os.path.join(root, file)
                    page_name = os.path.splitext(file)[0]
                    
                    pages.append({
                        'name': page_name,
                        'section': section,
                        'path': pdf_path,
                        'order': order
                    })
                    order += 1
        
        # Create structure file
        structure = {
            'notebook': nb_name,
            'pages': pages
        }
        
        structure_file = os.path.join(nb_path, '_structure.json')
        with open(structure_file, 'w', encoding='utf-8') as f:
            json.dump(structure, f, indent=2)
        
        print(f"  ✓ Created _structure.json with {len(pages)} pages\n")
    
    print(f"{'='*60}")
    print("✓ Structure files rebuilt successfully!")
    print("=" * 60)

elif has_structure:
    print(f"\n{'='*60}")
    print(f"✓ All {len(has_structure)} notebook(s) have valid structure files")
    print("=" * 60)

print(f"\nReady to process!")

In [ ]:
# Cell 3: MANUAL PATH OVERRIDE (if auto-detect failed)
# Uncomment and edit these lines if Cell 2 didn't find your files:

# PDF_DIR = "/content/drive/MyDrive/YOUR_FOLDER_HERE/notebooks_pdf"
# MD_DIR = "/content/drive/MyDrive/YOUR_FOLDER_HERE/notebooks_md"
# import os
# os.makedirs(MD_DIR, exist_ok=True)
# print(f"✅ PDF_DIR: {PDF_DIR}")
# print(f"✅ MD_DIR: {MD_DIR}")

In [ ]:
# Cell 4: Install Dependencies
!pip install -q marker-pdf pypdfium2 tqdm

In [ ]:
# Cell 5: Create Phase 2 Modules
import os

os.makedirs("phase2_ocr", exist_ok=True)

# pdf_slicer.py
with open("phase2_ocr/pdf_slicer.py", "w") as f:
    f.write('''import os
import tempfile
import pypdfium2 as pdfium

def get_page_count(pdf_path):
    doc = pdfium.PdfDocument(pdf_path)
    count = len(doc)
    doc.close()
    return count

def split_pdf(pdf_path):
    doc = pdfium.PdfDocument(pdf_path)
    num_pages = len(doc)
    if num_pages <= 1:
        doc.close()
        return [pdf_path]
    tmp_dir = tempfile.mkdtemp(dir="/tmp")
    page_paths = []
    for i in range(num_pages):
        single = pdfium.PdfDocument.new()
        single.import_pages(doc, [i])
        tmp_path = os.path.join(tmp_dir, f"page_{i:03d}.pdf")
        with open(tmp_path, "wb") as ff:
            single.save(ff)
        single.close()
        page_paths.append(tmp_path)
    doc.close()
    return page_paths

def cleanup_splits(page_paths, original_path):
    for p in page_paths:
        if p != original_path and os.path.exists(p):
            os.remove(p)
    if page_paths and page_paths[0] != original_path:
        tmp_dir = os.path.dirname(page_paths[0])
        try:
            os.rmdir(tmp_dir)
        except OSError:
            pass
''')

# marker_engine.py
with open("phase2_ocr/marker_engine.py", "w") as f:
    f.write('''import os
import contextlib
import torch
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.config.parser import ConfigParser
from phase2_ocr.pdf_slicer import split_pdf, cleanup_splits

class MarkerEngine:
    _instance = None
    _converter = None
    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance
    def _init_converter(self):
        if self._converter is not None:
            return
        print("Loading marker-pdf models...")
        config = ConfigParser({"output_format": "markdown", "force_ocr": False, "batch_size": 1})
        self._converter = PdfConverter(config=config.generate_config_dict(), artifact_dict=create_model_dict())
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Marker-pdf ready on: {device.upper()}")
    def convert(self, pdf_path: str) -> str:
        self._init_converter()
        page_paths = split_pdf(pdf_path)
        try:
            parts = []
            for p in page_paths:
                with open(os.devnull, "w") as devnull:
                    with contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):
                        result = self._converter(p)
                parts.append(result.markdown)
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            return "\\n\\n".join(parts)
        finally:
            cleanup_splits(page_paths, pdf_path)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

def convert_pdf_to_markdown(pdf_path: str) -> str:
    return MarkerEngine().convert(pdf_path)
''')

# utils.py
with open("phase2_ocr/utils.py", "w") as f:
    f.write('''import os
def find_notebook_folders(pdf_dir):
    notebooks = []
    if not os.path.exists(pdf_dir):
        return notebooks
    for item in os.listdir(pdf_dir):
        item_path = os.path.join(pdf_dir, item)
        if os.path.isdir(item_path):
            structure_file = os.path.join(item_path, "_structure.json")
            if os.path.exists(structure_file):
                notebooks.append({"folder": item_path, "name": item, "structure_file": structure_file})
    return notebooks

def safe_filename(notebook, section, page_name, order):
    def sanitize(s):
        return "".join(c if c.isalnum() or c in " -_" else "_" for c in s).strip()
    return f"{sanitize(notebook)[:50]}__{sanitize(section.replace('/', '_'))[:50]}__{order:03d}_{sanitize(page_name)[:50]}.md"

def create_md_header(pdf_path, notebook, section, page_name, order):
    return f"""---\nsource_pdf: {pdf_path}\nnotebook: {notebook}\nsection: {section}\npage: {page_name}\norder: {order}\n---\n\n# {page_name}\n\n"""
''')

with open("phase2_ocr/__init__.py", "w") as f:
    f.write("")

print("✓ Phase 2 modules created")

In [ ]:
# Cell 5.5: Fix Path Issues in Structure Files
import json
import os

print("=" * 60)
print("Checking and fixing paths in structure files...")
print("=" * 60)

from phase2_ocr.utils import find_notebook_folders

notebooks = find_notebook_folders(PDF_DIR)

if not notebooks:
    print("\n❌ No notebooks found with _structure.json files")
    print(f"Checked directory: {PDF_DIR}")
else:
    print(f"\nFound {len(notebooks)} notebook(s) with structure files\n")
    
    for nb in notebooks:
        print(f"Checking: {nb['name']}")
        
        with open(nb['structure_file'], 'r', encoding='utf-8') as f:
            structure = json.load(f)
        
        pages = structure.get('pages', [])
        print(f"  Structure contains {len(pages)} page(s)")
        
        # Check if paths need fixing
        if pages:
            first_path = pages[0]['path']
            print(f"  First PDF path: {first_path}")
            
            # If path is Windows-style, we need to fix it
            if '\\' in first_path or first_path.startswith('C:') or first_path.startswith('c:'):
                print(f"  ⚠ Windows paths detected - converting to Colab paths...")
                
                fixed_pages = []
                missing_count = 0
                
                for page in pages:
                    # Extract just the relative path after notebooks_pdf
                    old_path = page['path']
                    
                    # Find the part after "notebooks_pdf"
                    if 'notebooks_pdf' in old_path:
                        rel_part = old_path.split('notebooks_pdf', 1)[1]
                        rel_part = rel_part.replace('\\', '/')  # Convert backslashes
                        if rel_part.startswith('/'):
                            rel_part = rel_part[1:]
                        
                        # Build new Colab path
                        new_path = os.path.join(PDF_DIR, rel_part)
                        page['path'] = new_path
                        
                        if not os.path.exists(new_path):
                            missing_count += 1
                    
                    fixed_pages.append(page)
                
                structure['pages'] = fixed_pages
                
                # Save fixed structure
                with open(nb['structure_file'], 'w', encoding='utf-8') as f:
                    json.dump(structure, f, indent=2)
                
                print(f"  ✓ Fixed {len(fixed_pages)} page paths")
                
                if missing_count > 0:
                    print(f"  ⚠ Warning: {missing_count} PDF files not found at new paths")
                else:
                    print(f"  ✓ All PDF files found!")
            
            else:
                # Check if files exist
                exists_count = sum(1 for p in pages if os.path.exists(p['path']))
                print(f"  ✓ Paths look correct ({exists_count}/{len(pages)} files exist)")
        
        print()

print("=" * 60)
print("✓ Path check complete - ready to process!")
print("=" * 60)

In [ ]:
# Cell 6: Run OCR Processing
import sys, json, time, logging
from tqdm.notebook import tqdm
from phase2_ocr.marker_engine import convert_pdf_to_markdown
from phase2_ocr.utils import find_notebook_folders, safe_filename, create_md_header

def format_time(seconds):
    return f"{seconds:.0f}s" if seconds < 60 else f"{seconds/60:.1f}m" if seconds < 3600 else f"{seconds/3600:.1f}h"

def process_page(pdf_path, notebook, section, page_name, order):
    header = create_md_header(pdf_path, notebook, section, page_name, order)
    content = convert_pdf_to_markdown(pdf_path)
    return header + content

for name in ["marker", "surya", "texify", "PIL"]:
    logging.getLogger(name).setLevel(logging.WARNING)

print("=" * 60)
print("Phase 2: PDF to Markdown (Google Colab GPU)")
print("=" * 60)

notebooks = find_notebook_folders(PDF_DIR)
print(f"\nFound {len(notebooks)} notebook(s) with structure files")

if not notebooks:
    print("\n❌ No notebooks to process!")
    print("Make sure:")
    print("  1. PDFs are uploaded to Drive")
    print("  2. _structure.json files exist")
    print("  3. Paths in structure files are correct")
else:
    total_processed = 0
    total_skipped = 0
    total_missing = 0
    overall_start = time.time()
    
    for nb in notebooks:
        print(f"\n{'='*60}\nProcessing: {nb['name']}\n{'='*60}")
        
        with open(nb['structure_file'], 'r', encoding='utf-8') as f:
            structure = json.load(f)
        
        pages = structure.get('pages', [])
        notebook_name = structure.get('notebook', nb['name'])
        print(f"  Found {len(pages)} page(s) in structure file")
        
        nb_start = time.time()
        processed = 0
        skipped = 0
        missing = 0
        
        pbar = tqdm(pages, desc="  Converting", unit="page")
        
        for page_info in pbar:
            pdf_path = page_info['path']
            
            # Check if PDF exists
            if not os.path.exists(pdf_path):
                missing += 1
                total_missing += 1
                if missing <= 3:  # Show first 3 missing files
                    print(f"\n  ⚠ PDF not found: {pdf_path}")
                continue
            
            md_filename = safe_filename(notebook_name, page_info['section'], page_info['name'], page_info['order'])
            md_path = os.path.join(MD_DIR, md_filename)
            
            # Check if already processed
            if os.path.exists(md_path):
                skipped += 1
                total_skipped += 1
                continue
            
            try:
                # Actually process the page
                md_content = process_page(pdf_path, notebook_name, page_info['section'], page_info['name'], page_info['order'])
                
                # Write the output
                with open(md_path, 'w', encoding='utf-8') as f:
                    f.write(md_content)
                
                processed += 1
                total_processed += 1
                
                # Update progress bar
                avg_time = (time.time() - nb_start) / max(processed, 1)
                remaining = len(pages) - pbar.n - skipped - missing
                eta = avg_time * remaining
                pbar.set_postfix({'avg': f'{avg_time:.1f}s', 'eta': format_time(eta), 'done': processed})
            
            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    print(f"\n  ⚠ GPU OOM on {page_info['name']}, skipping...")
                    import torch
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                else:
                    print(f"\n  ✗ Error: {page_info['name']}: {e}")
            except Exception as e:
                print(f"\n  ✗ Error: {page_info['name']}: {e}")
        
        pbar.close()
        nb_time = time.time() - nb_start
        
        # Summary for this notebook
        print(f"\n  ✓ Processed: {processed}/{len(pages)} pages in {format_time(nb_time)}")
        if skipped > 0:
            print(f"  ⤭ Skipped: {skipped} already-processed pages")
        if missing > 0:
            print(f"  ⚠ Missing: {missing} PDF files not found")
        if processed > 0:
            print(f"  Average: {nb_time/processed:.1f}s per page")
    
    total_time = time.time() - overall_start
    print(f"\n{'='*60}\nProcessing Complete\n{'='*60}")
    print(f"Processed: {total_processed} pages")
    if total_skipped > 0:
        print(f"Skipped: {total_skipped} already-processed pages")
    if total_missing > 0:
        print(f"Missing: {total_missing} PDF files not found")
    print(f"Total time: {format_time(total_time)}")
    if total_processed > 0:
        print(f"Average: {total_time/total_processed:.1f}s per page")
    print(f"\nOutput directory: {MD_DIR}")

In [ ]:
# Cell 7: Verify Output
import os

print("=" * 60)
print("Verifying markdown output...")
print("=" * 60)

if not os.path.exists(MD_DIR):
    print(f"\n❌ Output directory doesn't exist: {MD_DIR}")
else:
    md_files = [f for f in os.listdir(MD_DIR) if f.endswith('.md')]
    
    print(f"\n✓ Output directory: {MD_DIR}")
    print(f"✓ Markdown files created: {len(md_files)}\n")
    
    if md_files:
        print("First 5 files created:")
        for f in sorted(md_files)[:5]:
            file_path = os.path.join(MD_DIR, f)
            size_kb = os.path.getsize(file_path) / 1024
            print(f"  • {f} ({size_kb:.1f} KB)")
        
        if len(md_files) > 5:
            print(f"  ... and {len(md_files) - 5} more")
    else:
        print("⚠ No markdown files created yet")
        print("\nPossible reasons:")
        print("  • All files already processed (check if .md files exist)")
        print("  • No PDFs found in structure")
        print("  • Processing errors occurred")